# Skyline Online Courses: Hypothesis Tests

Lesson 1.6 Practice Exercise. Runs t-tests on the Skyline Online Courses dataset,
demonstrates the multiple testing problem with simulation, and shows how
Bonferroni correction works.

Author: Kenya Harvey
Date: 08-13-2026

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [9]:
skyline = pd.read_csv("../lesson-1-2-types-of-data/skyline_enrollments.csv")

Python_for_Beginners = skyline[skyline["course_name"] == "Python for Beginners"]["hours_studied"]
SQL_basics = skyline[skyline["course_name"] == "SQL Basics"]["hours_studied"]

print(f"Python for Beginners: n={len(Python_for_Beginners)}, mean={Python_for_Beginners.mean():.2f}, std={Python_for_Beginners.std():.2f}")
print(f"SQL Basics: n={len(SQL_basics)}, mean={SQL_basics.mean():.2f}, std={SQL_basics.std():.2f}")

Python for Beginners: n=20, mean=30.71, std=3.30
SQL Basics: n=18, mean=28.91, std=2.04


In [4]:
t_stat, p_value = stats.ttest_ind(Python_for_Beginners, SQL_basics)

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 2.0016
p-value: 0.0529


In [10]:
mean_diff = Python_for_Beginners.mean() - SQL_basics.mean()

# Pooled standard error of the difference
n1, n2 = len(Python_for_Beginners), len(SQL_basics)
s1, s2 = Python_for_Beginners.std(ddof=1), SQL_basics.std(ddof=1)
se_diff = np.sqrt(s1**2 / n1 + s2**2 / n2)

# 95% CI for the difference
margin = 1.96 * se_diff
ci_lower = mean_diff - margin
ci_upper = mean_diff + margin

print(f"Mean difference (Python for Beginners - SQL Basics): {mean_diff:.2f}")
print(f"95% CI for the difference: ({ci_lower:.2f}, {ci_upper:.2f})")
print(f"p-value: {p_value:.4f}")

Mean difference (Python for Beginners - SQL Basics): 1.80
95% CI for the difference: (0.08, 3.53)
p-value: 0.0529


### Interpretation

The average difference in hours studied between Python for Beginners and SQL Basics was 3.45 hours. Because the p-value (\(p = 0.0529\)) is greater than the 0.05 significance threshold, this difference was not statistically detectable at the 5% level. The effect size was small, which means that this difference is not practically meaningful.

In [12]:
np.random.seed(2024)

n_tests = 10
significant_count = 0
all_p_values = []

for i in range(n_tests):
    # Two samples from the SAME distribution (null is true)
    a = np.random.normal(loc=100, scale=15, size=80)
    b = np.random.normal(loc=100, scale=15, size=80)

    _, p = stats.ttest_ind(a, b)
    all_p_values.append(p)

    if p < 0.05:
        significant_count += 1
        print(f"Test {i+1}: p = {p:.4f} (SIGNIFICANT, but null is actually true)")

print(f"\nOf {n_tests} tests on identical distributions, {significant_count} came back 'significant' at p < 0.05")

Test 7: p = 0.0193 (SIGNIFICANT, but null is actually true)

Of 10 tests on identical distributions, 1 came back 'significant' at p < 0.05


In [7]:
bonferroni_threshold = 0.05 / n_tests
significant_after_bonferroni = sum(p < bonferroni_threshold for p in all_p_values)

print(f"Original threshold: 0.05")
print(f"Bonferroni-corrected threshold: {bonferroni_threshold:.4f}")
print(f"Number significant at original threshold: {significant_count}")
print(f"Number significant after Bonferroni: {significant_after_bonferroni}")

Original threshold: 0.05
Bonferroni-corrected threshold: 0.0025
Number significant at original threshold: 1
Number significant after Bonferroni: 0


In [13]:
np.random.seed(0)

n_simulations = 500
n_tests_per_sim = 10

at_least_one_significant = 0

for _ in range(n_simulations):
    found_one = False
    for _ in range(n_tests_per_sim):
        a = np.random.normal(loc=100, scale=15, size=80)
        b = np.random.normal(loc=100, scale=15, size=80)
        _, p = stats.ttest_ind(a, b)
        if p < 0.05:
            found_one = True
            break
    if found_one:
        at_least_one_significant += 1

probability = at_least_one_significant / n_simulations
print(f"Of {n_simulations} simulations, each with {n_tests_per_sim} tests:")
print(f"At least one 'significant' result appeared in {at_least_one_significant} simulations")
print(f"Empirical probability of at least one false positive across 20 tests: {probability:.1%}")

Of 500 simulations, each with 10 tests:
At least one 'significant' result appeared in 206 simulations
Empirical probability of at least one false positive across 20 tests: 41.2%


## Part B: Multiple Testing in Practice

Of the 10 tests run on identical distributions, 1 came back “significant” at \(p < 0.05\). This was Test 7, with \(p = 0.0193\), which was significant even though the null hypothesis was actually true.